In [9]:
import pandas as pd
import numpy as np
import time

url = "https://archive.ics.uci.edu/static/public/597/productivity+prediction+of+garment+employees.zip"

df = pd.read_csv(url, compression="zip")

df.head()

,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382


In [15]:
df.shape

(1197, 16)

In [18]:
df.columns

Index(['date', 'quarter', 'department', 'day', 'team', 'targeted_productivity',
       'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men',
       'no_of_style_change', 'no_of_workers', 'actual_productivity',
       'MeetsTarget'],
      dtype='str')

In [11]:
df.isnull().sum()

date                       0
quarter                    0
department                 0
day                        0
team                       0
targeted_productivity      0
smv                        0
wip                      506
over_time                  0
incentive                  0
idle_time                  0
idle_men                   0
no_of_style_change         0
no_of_workers              0
actual_productivity        0
dtype: int64

In [12]:
df["wip"] = df["wip"].fillna(df["wip"].median())

In [13]:
df.isnull().sum()

date                     0
quarter                  0
department               0
day                      0
team                     0
targeted_productivity    0
smv                      0
wip                      0
over_time                0
incentive                0
idle_time                0
idle_men                 0
no_of_style_change       0
no_of_workers            0
actual_productivity      0
dtype: int64

In [14]:
df["MeetsTarget"] = (
    df["actual_productivity"] >= df["targeted_productivity"]
).astype(int)

df["MeetsTarget"].value_counts()

MeetsTarget
1    875
0    322
Name: count, dtype: int64

In [19]:
reg_target = "actual_productivity"
cls_target = "MeetsTarget"

drop_cols = ["actual_productivity", "MeetsTarget"]

X = df.drop(columns=drop_cols)
y_reg = df[reg_target]
y_cls = df[cls_target]

print("Features:", X.shape)
print("Regression target:", y_reg.shape)
print("Classification target:", y_cls.shape)
print("\nFeature columns:")
print(X.columns.tolist())

Features: (1197, 14)
Regression target: (1197,)
Classification target: (1197,)

Feature columns:
['date', 'quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']


In [24]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(df))

train_idx, test_idx = train_test_split(
    idx,
    test_size=0.2,
    random_state=42
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_reg_train = y_reg.iloc[train_idx].copy()
y_reg_test = y_reg.iloc[test_idx].copy()

y_cls_train = y_cls.iloc[train_idx].copy()
y_cls_test = y_cls.iloc[test_idx].copy()

print("Training samples:", len(train_idx))
print("Testing samples:", len(test_idx))
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

Training samples: 957
Testing samples: 240
X_train: (957, 14)
X_test: (240, 14)


In [25]:
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", cat_cols)
print("Numerical columns:", num_cols)

Categorical columns: ['date', 'quarter', 'department', 'day']
Numerical columns: ['team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']


C:\Users\Ummehani Khatri\AppData\Local\Temp\ipykernel_18620\2168273457.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()


In [26]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_cols = X_train.select_dtypes(include=["str"]).columns.tolist()
num_cols = X_train.select_dtypes(exclude=["str"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)

print("Before preprocessing:", X_train.shape)
print("After preprocessing:", X_train_p.shape)

Before preprocessing: (957, 14)
After preprocessing: (957, 83)


In [27]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

reg_model = LinearRegression()

start = time.perf_counter()
reg_model.fit(X_train_p, y_reg_train)
reg_train_time = time.perf_counter() - start

start = time.perf_counter()
reg_pred = reg_model.predict(X_test_p)
reg_pred_time = time.perf_counter() - start

reg_mae = mean_absolute_error(y_reg_test, reg_pred)
reg_rmse = np.sqrt(mean_squared_error(y_reg_test, reg_pred))
reg_r2 = r2_score(y_reg_test, reg_pred)

print("Regression Results")
print("MAE :", reg_mae)
print("RMSE:", reg_rmse)
print("R2  :", reg_r2)
print("Training time :", reg_train_time)
print("Prediction time:", reg_pred_time)

Regression Results
MAE : 0.11209626834622172
RMSE: 0.1506451206155621
R2  : 0.14531660005712477
Training time : 0.010311300000012125
Prediction time: 0.0002974999997604755


In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

cls_model = LogisticRegression(max_iter=1000)

start = time.perf_counter()
cls_model.fit(X_train_p, y_cls_train)
cls_train_time = time.perf_counter() - start

start = time.perf_counter()
cls_pred = cls_model.predict(X_test_p)
cls_pred_time = time.perf_counter() - start

cls_acc = accuracy_score(y_cls_test, cls_pred)
cls_precision = precision_score(y_cls_test, cls_pred)
cls_recall = recall_score(y_cls_test, cls_pred)
cls_f1 = f1_score(y_cls_test, cls_pred)

print("Classification Results")
print("Accuracy :", cls_acc)
print("Precision:", cls_precision)
print("Recall   :", cls_recall)
print("F1-score :", cls_f1)
print("Training time :", cls_train_time)
print("Prediction time:", cls_pred_time)

Classification Results
Accuracy : 0.7583333333333333
Precision: 0.7767441860465116
Recall   : 0.943502824858757
F1-score : 0.8520408163265306
Training time : 0.030429100000219478
Prediction time: 0.00039739999920129776


PART - B

In [29]:
cat_cols = ["date", "quarter", "department", "day"]
num_cols = [
    "team", "targeted_productivity", "smv", "wip",
    "over_time", "incentive", "idle_time", "idle_men",
    "no_of_style_change", "no_of_workers"
]

train_cat = pd.get_dummies(X_train[cat_cols], dtype=float)
test_cat = pd.get_dummies(X_test[cat_cols], dtype=float)

train_cat, test_cat = train_cat.align(
    test_cat,
    join="left",
    axis=1,
    fill_value=0
)

tr_num = X_train[num_cols].copy()
te_num = X_test[num_cols].copy()

mean = tr_num.mean()
std = tr_num.std()

tr_num = (tr_num - mean) / std
te_num = (te_num - mean) / std

X_train_m = np.hstack([
    tr_num.to_numpy(),
    train_cat.to_numpy()
])

X_test_m = np.hstack([
    te_num.to_numpy(),
    test_cat.to_numpy()
])

X_train_m = np.c_[np.ones(X_train_m.shape[0]), X_train_m]
X_test_m = np.c_[np.ones(X_test_m.shape[0]), X_test_m]

print("Manual training data:", X_train_m.shape)
print("Manual testing data :", X_test_m.shape)

Manual training data: (957, 84)
Manual testing data : (240, 84)


In [30]:
start = time.perf_counter()

xtx = X_train_m.T @ X_train_m
xty = X_train_m.T @ y_reg_train.to_numpy()

coef = np.linalg.pinv(xtx) @ xty

reg_m_train_time = time.perf_counter() - start

start = time.perf_counter()
reg_m_pred = X_test_m @ coef
reg_m_pred_time = time.perf_counter() - start

err = y_reg_test.to_numpy() - reg_m_pred

reg_m_mae = np.mean(np.abs(err))
reg_m_rmse = np.sqrt(np.mean(err ** 2))

ss_res = np.sum(err ** 2)
ss_tot = np.sum((y_reg_test.to_numpy() - np.mean(y_reg_test.to_numpy())) ** 2)
reg_m_r2 = 1 - (ss_res / ss_tot)

print("Manual Regression Results")
print("MAE :", reg_m_mae)
print("RMSE:", reg_m_rmse)
print("R2  :", reg_m_r2)
print("Training time :", reg_m_train_time)
print("Prediction time:", reg_m_pred_time)

Manual Regression Results
MAE : 0.11209630610472095
RMSE: 0.1506450997886165
R2  : 0.14531683637999204
Training time : 0.013177900000300724
Prediction time: 3.550000019458821e-05


In [31]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

xtr = X_train_m
xte = X_test_m
ytr = y_cls_train.to_numpy()
yte = y_cls_test.to_numpy()

w = np.zeros(xtr.shape[1])

lr = 0.01
epochs = 5000

start = time.perf_counter()

for _ in range(epochs):
    p = sigmoid(xtr @ w)
    grad = (xtr.T @ (p - ytr)) / len(ytr)
    w -= lr * grad

cls_m_train_time = time.perf_counter() - start

start = time.perf_counter()

prob = sigmoid(xte @ w)
cls_m_pred = (prob >= 0.5).astype(int)

cls_m_pred_time = time.perf_counter() - start

tp = np.sum((yte == 1) & (cls_m_pred == 1))
tn = np.sum((yte == 0) & (cls_m_pred == 0))
fp = np.sum((yte == 0) & (cls_m_pred == 1))
fn = np.sum((yte == 1) & (cls_m_pred == 0))

cls_m_acc = (tp + tn) / len(yte)
cls_m_precision = tp / (tp + fp) if tp + fp else 0
cls_m_recall = tp / (tp + fn) if tp + fn else 0
cls_m_f1 = (
    2 * cls_m_precision * cls_m_recall /
    (cls_m_precision + cls_m_recall)
    if cls_m_precision + cls_m_recall else 0
)

print("Manual Classification Results")
print("Accuracy :", cls_m_acc)
print("Precision:", cls_m_precision)
print("Recall   :", cls_m_recall)
print("F1-score :", cls_m_f1)
print("Training time :", cls_m_train_time)
print("Prediction time:", cls_m_pred_time)

Manual Classification Results
Accuracy : 0.75
Precision: 0.7577092511013216
Recall   : 0.9717514124293786
F1-score : 0.8514851485148516
Training time : 0.12207190000026458
Prediction time: 0.00010410000049887458


In [32]:
comparison = pd.DataFrame({
    "Model": [
        "Sklearn Linear Regression",
        "Manual Linear Regression",
        "Sklearn Logistic Regression",
        "Manual Logistic Regression"
    ],
    "MAE": [
        reg_mae,
        reg_m_mae,
        np.nan,
        np.nan
    ],
    "RMSE": [
        reg_rmse,
        reg_m_rmse,
        np.nan,
        np.nan
    ],
    "R2": [
        reg_r2,
        reg_m_r2,
        np.nan,
        np.nan
    ],
    "Accuracy": [
        np.nan,
        np.nan,
        cls_acc,
        cls_m_acc
    ],
    "Precision": [
        np.nan,
        np.nan,
        cls_precision,
        cls_m_precision
    ],
    "Recall": [
        np.nan,
        np.nan,
        cls_recall,
        cls_m_recall
    ],
    "F1": [
        np.nan,
        np.nan,
        cls_f1,
        cls_m_f1
    ],
    "Train Time": [
        reg_train_time,
        reg_m_train_time,
        cls_train_time,
        cls_m_train_time
    ],
    "Prediction Time": [
        reg_pred_time,
        reg_m_pred_time,
        cls_pred_time,
        cls_m_pred_time
    ]
})

comparison

,Model,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Train Time,Prediction Time
0,Sklearn Linear Regression,0.112096,0.150645,0.145317,NaN,NaN,NaN,NaN,0.010311,0.000297
1,Manual Linear Regression,0.112096,0.150645,0.145317,NaN,NaN,NaN,NaN,0.013178,0.000036
2,Sklearn Logistic Regression,NaN,NaN,NaN,0.758333,0.776744,0.943503,0.852041,0.030429,0.000397
3,Manual Logistic Regression,NaN,NaN,NaN,0.750000,0.757709,0.971751,0.851485,0.122072,0.000104


In [33]:
start = time.perf_counter()

w_opt = np.zeros(X_train_m.shape[1])

lr_opt = 0.05
epochs_opt = 2000

for _ in range(epochs_opt):
    p = sigmoid(X_train_m @ w_opt)
    grad = X_train_m.T @ (p - y_cls_train.to_numpy()) / len(y_cls_train)
    w_opt -= lr_opt * grad

opt_train_time = time.perf_counter() - start

start = time.perf_counter()

opt_prob = sigmoid(X_test_m @ w_opt)
opt_pred = (opt_prob >= 0.5).astype(int)

opt_pred_time = time.perf_counter() - start

y = y_cls_test.to_numpy()

tp = np.sum((y == 1) & (opt_pred == 1))
tn = np.sum((y == 0) & (opt_pred == 0))
fp = np.sum((y == 0) & (opt_pred == 1))
fn = np.sum((y == 1) & (opt_pred == 0))

opt_acc = (tp + tn) / len(y)
opt_precision = tp / (tp + fp) if tp + fp else 0
opt_recall = tp / (tp + fn) if tp + fn else 0
opt_f1 = (
    2 * opt_precision * opt_recall /
    (opt_precision + opt_recall)
    if opt_precision + opt_recall else 0
)

print("Optimized Manual Logistic Regression")
print("Accuracy :", opt_acc)
print("Precision:", opt_precision)
print("Recall   :", opt_recall)
print("F1-score :", opt_f1)
print("Training time :", opt_train_time)
print("Prediction time:", opt_pred_time)

Optimized Manual Logistic Regression
Accuracy : 0.7625
Precision: 0.7727272727272727
Recall   : 0.96045197740113
F1-score : 0.8564231738035264
Training time : 0.05619970000043395
Prediction time: 7.7699999565084e-05


In [34]:
final_comparison = pd.DataFrame({
    "Model": [
        "Sklearn Linear Regression",
        "Manual Linear Regression",
        "Sklearn Logistic Regression",
        "Manual Logistic Regression",
        "Optimized Manual Logistic Regression"
    ],
    "MAE": [
        reg_mae,
        reg_m_mae,
        np.nan,
        np.nan,
        np.nan
    ],
    "RMSE": [
        reg_rmse,
        reg_m_rmse,
        np.nan,
        np.nan,
        np.nan
    ],
    "R2": [
        reg_r2,
        reg_m_r2,
        np.nan,
        np.nan,
        np.nan
    ],
    "Accuracy": [
        np.nan,
        np.nan,
        cls_acc,
        cls_m_acc,
        opt_acc
    ],
    "Precision": [
        np.nan,
        np.nan,
        cls_precision,
        cls_m_precision,
        opt_precision
    ],
    "Recall": [
        np.nan,
        np.nan,
        cls_recall,
        cls_m_recall,
        opt_recall
    ],
    "F1": [
        np.nan,
        np.nan,
        cls_f1,
        cls_m_f1,
        opt_f1
    ],
    "Train Time": [
        reg_train_time,
        reg_m_train_time,
        cls_train_time,
        cls_m_train_time,
        opt_train_time
    ],
    "Prediction Time": [
        reg_pred_time,
        reg_m_pred_time,
        cls_pred_time,
        cls_m_pred_time,
        opt_pred_time
    ]
})

final_comparison.round(6)

,Model,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Train Time,Prediction Time
0,Sklearn Linear Regression,0.112096,0.150645,0.145317,NaN,NaN,NaN,NaN,0.010311,0.000297
1,Manual Linear Regression,0.112096,0.150645,0.145317,NaN,NaN,NaN,NaN,0.013178,0.000036
2,Sklearn Logistic Regression,NaN,NaN,NaN,0.758333,0.776744,0.943503,0.852041,0.030429,0.000397
3,Manual Logistic Regression,NaN,NaN,NaN,0.750000,0.757709,0.971751,0.851485,0.122072,0.000104
4,Optimized Manual Logistic Regression,NaN,NaN,NaN,0.762500,0.772727,0.960452,0.856423,0.056200,0.000078


In [35]:
final_comparison.round(6).to_csv(
    "model_comparison.csv",
    index=False
)

print("model_comparison.csv saved successfully")

model_comparison.csv saved successfully
